# Step 08 — Attach Building Geometry to LLM Predictions

**Input:**
- `data/output/llm_predictions/predictions_checkpoint.parquet`
- `data/output/05_condensed_buildings_with_pois.gpkg` (geometry source)

**Output:** `data/output/llm_predictions/predictions_with_geometry.gpkg`

From the linux branch: clean two-step workflow that joins flat parquet predictions back to building geometry.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import pandas as pd
import geopandas as gpd
print('Config loaded')

In [ ]:
predictions = pd.read_parquet(LLM_CHECKPOINT_FILE)
print(f'Loaded {len(predictions):,} predictions')

geom_source = gpd.read_file(CONDENSED_BUILDINGS_FILE)
geom_source['gml_id'] = geom_source['gml_id'].astype(str)
predictions['gml_id'] = predictions['gml_id'].astype(str)

merged = predictions.merge(geom_source[['gml_id', 'geometry']], on='gml_id', how='left')
gdf    = gpd.GeoDataFrame(merged, geometry='geometry', crs=geom_source.crs)

no_geom = gdf['geometry'].isna().sum()
if no_geom > 0:
    print(f'WARNING: {no_geom} predictions have no geometry (gml_id not found in source)')

out_path = LLM_PREDICTIONS_DIR / 'predictions_with_geometry.gpkg'
gdf.to_file(out_path, driver='GPKG')
print(f'Saved {len(gdf):,} rows → {out_path}')